In [63]:
from xaikd import models, datasets, utils
from xaikd.utils import metrics
import torch
from torch.utils.data import random_split, DataLoader
from torch import nn
from torch.nn import functional as F
from datetime import datetime
import numpy as np

from tqdm.notebook import tqdm

from numpy import typing as npt

import os

import pandas as pd

In [45]:
DATASET_NAME = "cifar100-people"
DATASIZE = 0.1
DEVICE = utils.get_device()

DEVICE

'cuda'

In [2]:
dataset = datasets.construct("cifar100-people")

In [16]:
trng = torch.Generator()
trng.manual_seed(1)

ds_train = dataset.create_subset(train_split=True)

ds_train, _ = random_split(
    ds_train,
    [DATASIZE, 1.0-DATASIZE],
    generator=trng
)

ds_val = dataset.create_subset(train_split=False)

dl_train = datasets.build_dataloader(ds_train, shuffle=False)
dl_val = datasets.build_dataloader(ds_val, shuffle=False)

In [14]:
model = models.get_trained_model("cifar100-resnet18-v1")
utils.modify_last_layer_for_subclasses(model, dataset.selected_classes)

model.to(DEVICE);

In [19]:
metrics.accuracy(
    model=model,
    dataloader=dl_val,
    num_classes=len(dataset.selected_classes),
    device=DEVICE,
)

(0.6200000047683716, 1.2948641777038574)

# Extract Activation and Grad

In [140]:
class OutputQuantity:
    def __call__(self, logits, target_logits):
        raise NotImplementedError()
    def __str__(self):
        return self.__class__.__name__


def compute_log_odd_winning(logits: torch.Tensor) -> torch.Tensor:

    ns, nc = logits.shape

    values, _ = torch.topk(logits, dim=1, k=nc)

    logit_winning = values[:, 0]
    lse_others = torch.logsumexp(values[:, 1:], dim=1)
    log_odd = logit_winning - lse_others

    return log_odd

        
class LogOddWinningClass(OutputQuantity):
    def __call__(self, logits, targets):
        return torch.sign(logits).detach() * logits
        return compute_log_odd_winning(logits)



class LogitWinningClass(OutputQuantity):
    def __call__(self, logits, targets):
        values, _ = torch.topk(logits, dim=1, k=1)

        logit_winning = values[:, 0]
        return logit_winning


class SumAllLogits(OutputQuantity):
    def __call__(self, logits, targets):
        return logits.sum(dim=1)
        

def extract_activation_context(
    model: nn.Module,
    layer: str,
    data_loader: DataLoader,
    output_quantity: OutputQuantity,
    rng: np.random.Generator,
    device="cpu",
    number_of_selected_spatial_locations=20,
    strict_mode=False,
    verbose=False,
):
    arr_act = []
    arr_ctx = []

    try:
        module, hook = utils.interceptor.attach_hook_intercept_layer_output(
            model, layer, should_retain_grad=True, detach_output=False
        )

        for batch in tqdm(data_loader, desc=f"extract act ctx output_quantity={output_quantity.__class__.__name__}) at layer={layer}"):
            x, y = batch
            x = x.to(device)


            logits = model(x)
            (output_quantity(logits, y)).sum().backward()
            act = utils.interceptor.get_output(module)
            output_dimensions = act.shape[1:]

            assert act.grad is not None
            ctx = act.grad

            assert ctx.shape == act.shape
            
            act = act.detach().cpu().numpy()
            ctx = ctx.detach().cpu().numpy()

            selected_act, selected_ctx = utils.subsample_tensors(
                act,
                ctx,
                num_locations=number_of_selected_spatial_locations,
                rng=rng,
            )
            arr_act.append(selected_act)
            arr_ctx.append(selected_ctx)

    finally:
        hook.remove()

    print(f"{layer}: output-dims={output_dimensions}")

    arr_act = np.vstack(arr_act)
    arr_ctx = np.vstack(arr_ctx)

    return arr_act, arr_ctx


def extract_act_grad(model, layer, output_quantity, dl):
    


    return extract_activation_context(
        model=model,
        layer=layer,
        data_loader=dl,
        output_quantity=output_quantity,
        rng=np.random.default_rng(seed=1),
        device=DEVICE,
    )

extract_act_grad(model, "avgpool", SumAllLogits(), dl_train)

extract act ctx output_quantity=SumAllLogits) at layer=avgpool:   0%|          | 0/4 [00:00<?, ?it/s]

avgpool: output-dims=torch.Size([512, 1, 1])


(array([[0.10102677, 0.960209  , 0.3827793 , ..., 0.0601557 , 0.5523869 ,
         0.37362525],
        [0.30660984, 2.3266697 , 0.03504136, ..., 0.08452825, 0.01399533,
         0.04893738],
        [0.01976126, 0.7626184 , 0.00302856, ..., 0.04187122, 0.12033181,
         0.00557438],
        ...,
        [0.04629413, 1.1809896 , 0.42448714, ..., 0.20044588, 0.01310524,
         0.47162923],
        [0.08555653, 1.8575144 , 0.        , ..., 0.        , 0.3009705 ,
         0.09433806],
        [0.05903496, 1.4818411 , 1.5698389 , ..., 0.10980165, 1.6403408 ,
         0.3017176 ]], dtype=float32),
 array([[-0.08207521,  0.96054804,  0.00844356, ..., -0.11760008,
         -0.03609873,  0.03940967],
        [-0.08207521,  0.96054804,  0.00844356, ..., -0.11760008,
         -0.03609873,  0.03940967],
        [-0.08207521,  0.96054804,  0.00844356, ..., -0.11760008,
         -0.03609873,  0.03940967],
        ...,
        [-0.08207521,  0.96054804,  0.00844356, ..., -0.11760008,
         

# Construing Basis

In [50]:
def _solve_eigvecs(cov, sort_func=lambda x: x):
    eigvals, eigvecs = np.linalg.eigh(cov)

    assert len(eigvals.shape) == 1

    indices = np.argsort(-sort_func(eigvals))
    eigvals = eigvals[indices]
    eigvecs = eigvecs[:, indices]

    return eigvecs
    
class BasisInterface:
    def get_Uk(self, k: int) -> npt.NDArray:
        raise NotImplementedError()   

class PCA(BasisInterface):
    def __init__(self, arr_act, arr_ctx, layer=None, task_id=None):
        cov = arr_act.T @ arr_act
        self.U = _solve_eigvecs(cov)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]


class GradPCA(BasisInterface):
    def __init__(self, arr_act, arr_ctx, layer=None, task_id=None):
        cov = arr_ctx.T @ arr_ctx
        self.U = _solve_eigvecs(cov)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]

class PRCASortAbs(BasisInterface):
    def __init__(self, arr_act, arr_ctx, layer=None, task_id=None):
        cov = arr_act.T @ arr_ctx + arr_ctx.T @ arr_act
        self.U = _solve_eigvecs(cov, sort_func=np.abs)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]


class PRCA(BasisInterface):
    def __init__(self, arr_act, arr_ctx, layer=None, task_id=None):
        cov = arr_act.T @ arr_ctx + arr_ctx.T @ arr_act
        self.U = _solve_eigvecs(cov)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]


class PRCASignAlignSortAbs(BasisInterface):
    def __init__(self, arr_act, arr_ctx, layer=None, task_id=None):
        arr_rel = (arr_act * arr_ctx).sum(axis=1, keepdims=True)
        arr_ctx = (arr_rel >= 0) * arr_ctx - (arr_rel < 0) * arr_ctx
        
        cov = arr_act.T @ arr_ctx + arr_ctx.T @ arr_act
        
        self.U = _solve_eigvecs(cov, sort_func=np.abs)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]

In [90]:
from xaikd.bases import learners
def exponential_map(p, g, t):
    norm_g = torch.linalg.norm(g)
    return torch.cos(norm_g * t) * p + torch.sin(norm_g * t) * (g / (norm_g + 1e-16))
    
class PRCAReconGreedyLearnerDev:
    def fit(
        self,
        activation: npt.NDArray,
        context: npt.NDArray,
        epochs=100,
        eps=1e-6,
        seed=1,
        largest_k=None,
        device="cpu",
    ) -> npt.NDArray:
        assert activation.shape == context.shape

        nb, d = activation.shape

        activation = activation / ((np.mean(activation**2) ** (1 / 2)) * (d ** (1 / 4)))
        context = context / ((np.mean(context**2) ** (1 / 2)) * (d ** (1 / 4)))

        activation = torch.from_numpy(activation).float().to(device)
        context = torch.from_numpy(context).float().to(device)

        rng = torch.Generator()
        rng.manual_seed(seed)

        U = torch.zeros(d, d)
        U = U.to(device)

        I = torch.eye(d).to(device)

        if largest_k is None:
            largest_k = d

        tbar = tqdm(
            range(largest_k),
            total=largest_k,
            desc=f"PRCA Recon [device={device}]]",
        )

        for k in tbar:
            if k >= 1:
                Uk = U[:, :k]
                U_pinv = torch.linalg.pinv(Uk)
                U_complement = I - U_pinv.T @ Uk.T
            else:
                U_pinv = 0
                U_complement = I

            v = torch.randn(d, generator=rng).to(device)
            v = v / torch.linalg.norm(v)

            v = v.to(device)

            # take only activation and context that are not captured by the learned directions
            activation_on_U_comp = activation @ U_complement
            context_on_U_comp = context @ U_complement

            ref_rel = (activation_on_U_comp * context_on_U_comp).sum(dim=1, keepdims=True)
            assert ref_rel.shape == (nb, 1)
            
            for _ in range(epochs):
                v.requires_grad_(True)
                v.grad = None

                obj = self._obj_func(activation, context, v, ref_rel)

                tbar.set_description_str(
                    f"PRCAReconGreedy [device={device}] obj={obj.detach().cpu().numpy():.4e}"
                )

                obj.backward()

                with torch.no_grad():
                    ov = v

                    grad = v.grad
                    grad = (I - torch.outer(v, v)) @ grad

                    @torch.no_grad()
                    def linesearch(p, g):
                        # todo: separate this from function
                        max_alpha = 2 * np.pi / torch.linalg.norm(g)

                        arr_steps = torch.linspace(0, max_alpha, 100).to(device)
                        norm_g = torch.linalg.norm(g)

                        # construct candidate from exponential map at different time steps
                        term_cos = torch.outer(p, torch.cos(norm_g * arr_steps))
                        term_sin = torch.outer(
                            (g / norm_g), torch.sin(norm_g * arr_steps)
                        )

                        arr_directions = term_cos + term_sin


                        rel_on_dir = (activation @ arr_directions) * (
                            context @ arr_directions
                        )

                        arr_obj_directions = (
                            -torch.mean((ref_rel - rel_on_dir) ** 2, axis=0)
                            .detach()
                            .cpu()
                            .numpy()
                        )

                        assert arr_obj_directions.shape == (
                            arr_steps.shape[0],
                        ), arr_obj_directions.shape

                        best_ix = np.argmax(arr_obj_directions)
                        best_lr = arr_steps[best_ix].detach().cpu()

                        return best_lr

                    if torch.linalg.norm(grad).detach().cpu().numpy() == 0:
                        break

                    lr = linesearch(v, grad)

                    assert lr >= 0

                    if lr == 0:
                        break

                    # update v with gradient `ascent`.
                    v = exponential_map(v, grad, lr)

                    # theorethically, the exponential map should return a vector with unitnorm,
                    # but sometimes there is some numerical instability.
                    v = v / torch.linalg.norm(v)

                    np.testing.assert_allclose(
                        torch.linalg.norm(v).detach().cpu(), 1.0, atol=1e-3
                    )

                    if (v @ ov).abs() > (1 - eps):
                        # stop if the solution isn't update anymore.
                        break
            print(f"[k={k}] loss={obj.detach().cpu().numpy():.4f}")
            U[:, k] = v.detach()

        with torch.no_grad():

            Q, R = torch.linalg.qr(U)
            U = Q
        np.testing.assert_allclose(
            (U.T @ U).detach().cpu().numpy(), np.eye(d), atol=1e-6
        )

        return U.detach().cpu().numpy()

    @staticmethod
    def _obj_func(
        activation: torch.Tensor,
        context: torch.Tensor,
        u: torch.Tensor,
        ref_rel: torch.Tensor
    ) -> torch.Tensor:

        activation_projected = activation.matmul(u)
        context_projected = context.matmul(u)

        assert len(activation_projected.shape) == len(context_projected.shape) == 1

        relevance_original = ref_rel.squeeze(dim=1)
        relevance_projected = activation_projected * context_projected
        assert relevance_original.shape == relevance_projected.shape

        obj = (relevance_original - relevance_projected) ** 2

        assert len(obj.shape) == 1 and obj.shape[0] == activation.shape[0]

        # convert the problem into maximization problem.
        loss = -obj.mean()

        return loss

class PRCAReconGreedy(BasisInterface):
    def __init__(self, arr_act, arr_ctx, layer=None, task_id=None):
        cov = arr_act.T @ arr_ctx + arr_ctx.T @ arr_act
        self.U =  PRCAReconGreedyLearnerDev().fit(
            activation=arr_act, context=arr_ctx, device=DEVICE
        )
        
    def get_Uk(self, k: int):
        return self.U[:, :k]

In [91]:
class PRCAReconNonGreedyLearner:
    def __init__(self, task_id, layer):
        self.task_id = task_id
        self.layer = layer
    def fit(
        self,  arr_act: npt.NDArray, arr_ctx: npt.NDArray, k: int,  U_init=None, device="cpu",
    ) -> npt.NDArray:
        n, d, = arr_act.shape

        assert arr_ctx.shape == arr_act.shape, arr_ctx.shape
        
        lr = 1e-4
        epochs = 5000
                
        n, d, = arr_act.shape

        scale_act = ((np.mean(arr_act**2) ** (1 / 2)) * (d ** (1 / 4)))
        scale_ctx = ((np.mean(arr_ctx**2) ** (1 / 2)) * (d ** (1 / 4)))
        arr_act = arr_act / scale_act
        arr_ctx = arr_ctx / scale_ctx
        
        arr_act: torch.Tensor = torch.from_numpy(arr_act).to(device)
        arr_ctx: torch.Tensor = torch.from_numpy(arr_ctx).to(device)


        linear_layer = torch.nn.Linear(k, d, bias=False)
        trng = torch.Generator()
        trng.manual_seed(1)
        if U_init is None:
            U_init = torch.randn((k, d), generator=trng)
        else:
            U_init = torch.from_numpy(U_init.T)
            
        linear_layer.weight = torch.nn.Parameter(U_init)
    
        ortho_layer = torch.nn.utils.parametrizations.orthogonal(linear_layer).to(device)
        assert ortho_layer.weight.shape == (k, d)
        
        optimizer = torch.optim.Adam(ortho_layer.parameters(), lr=lr)

        rel = (arr_act*arr_ctx).sum(dim=1)
        
        pgb = tqdm(range(epochs), desc=f"{self.__class__.__name__} (k={k})")
        for epoch in pgb:
            optimizer.zero_grad()
            
            # shape: (k, d)
            U = ortho_layer.weight

            act_proj = (arr_act @ U.T ) @ U

            ctx_proj = (arr_ctx @ U.T) @ U
            
            rel_recon = (act_proj * ctx_proj).sum(dim=1)


            # shape = (n, )
            loss = (rel - rel_recon).pow(2)
    
            loss = loss.mean()
            
            loss.backward()
        
            optimizer.step()

            loss = loss.detach().cpu().numpy()
                
            pgb.set_description_str(f"{self.__class__.__name__} (k={k}; lr={lr}) loss={loss:.4e}")
            
        U =  ortho_layer.weight.T.detach().cpu().numpy()
        
        # sanity_check
        np.testing.assert_allclose(U.T @ U, np.eye(k), atol=1e-4)

        return U


class PRCARecon(BasisInterface):
    def __init__(self,  arr_act, arr_ctx, layer=None, task_id=None):
        self.arr_act = arr_act
        self.arr_ctx = arr_ctx
        self.layer = layer
        self.task_id = task_id
        self.is_slow = True
        
    def get_Uk(self, k: int):

        return PRCAReconNonGreedyLearner(
            layer=self.layer,
            task_id=self.task_id
        ).fit(
            self.arr_act, self.arr_ctx, 
            k=k,
            U_init=PCA(arr_act=self.arr_act, arr_ctx=self.arr_ctx).get_Uk(k),
            device=DEVICE
        )

## Full Jacobian

In [115]:
class LogitOfOutput(OutputQuantity):
    def __init__(self, output_ix):
        self.output_ix = output_ix
    def __call__(self, logits, targets):
        return logits[:, self.output_ix]


def extract_act_jacobian(
    model,
    layer,
    dl,
    device=DEVICE
):
    arr_acts = None
    arr_arr_ctxs = []
    
    for cix in range(len(dataset.selected_classes)):
        rng = np.random.default_rng(seed=1)

        _arr_acts, _arr_ctx = extract_activation_context(
            model=model,
            layer=layer,
            data_loader=dl,
            output_quantity=LogitOfOutput(output_ix=cix),
            rng=rng,
            device=DEVICE,
        )
        if arr_acts is None:
            arr_acts = _arr_acts
        else:
            np.testing.assert_allclose(arr_acts, _arr_acts)

        arr_arr_ctxs.append(_arr_ctx)


    arr_arr_ctxs = np.stack(arr_arr_ctxs)
    arr_arr_ctxs = np.transpose(arr_arr_ctxs, [1, 0, 2])
    

    assert arr_arr_ctxs.shape == (arr_acts.shape[0], len(dataset.selected_classes), arr_acts.shape[1])

    return arr_acts, arr_arr_ctxs
    
def ano():
    extract_act_jacobian(
        model, "layer4", dl_train
    )
    print("sanity-check passed")
ano()

extract act ctx output_quantity=LogitOfOutput) at layer=layer4:   0%|          | 0/4 [00:00<?, ?it/s]

layer4: output-dims=torch.Size([512, 4, 4])


extract act ctx output_quantity=LogitOfOutput) at layer=layer4:   0%|          | 0/4 [00:00<?, ?it/s]

layer4: output-dims=torch.Size([512, 4, 4])


extract act ctx output_quantity=LogitOfOutput) at layer=layer4:   0%|          | 0/4 [00:00<?, ?it/s]

layer4: output-dims=torch.Size([512, 4, 4])


extract act ctx output_quantity=LogitOfOutput) at layer=layer4:   0%|          | 0/4 [00:00<?, ?it/s]

layer4: output-dims=torch.Size([512, 4, 4])


extract act ctx output_quantity=LogitOfOutput) at layer=layer4:   0%|          | 0/4 [00:00<?, ?it/s]

layer4: output-dims=torch.Size([512, 4, 4])
sanity-check passed


In [174]:
class PRCAReconNonGreedyFullJacobianLearner:
    def __init__(self, task_id, layer):
        self.task_id = task_id
        self.layer = layer
    def fit(
        self,  arr_act: npt.NDArray, arr_ctx: npt.NDArray, k: int,  U_init=None, device="cpu",
    ) -> npt.NDArray:
        n, d, = arr_act.shape
        
        lr = 1e-3
        epochs = 5000
                
        n, d, = arr_act.shape

        scale_act = ((np.mean(arr_act**2) ** (1 / 2)) * (d ** (1 / 4)))
        scale_ctx = ((np.mean(arr_ctx**2) ** (1 / 2)) * (d ** (1 / 4)))
        arr_act = arr_act / scale_act
        arr_ctx = arr_ctx / scale_ctx
        
        arr_act: torch.Tensor = torch.from_numpy(arr_act).to(device)
        arr_ctx: torch.Tensor = torch.from_numpy(arr_ctx).to(device)


        linear_layer = torch.nn.Linear(k, d, bias=False)
        trng = torch.Generator()
        trng.manual_seed(1)
        if U_init is None:
            U_init = torch.randn((k, d), generator=trng)
    
        else:
            U_init = torch.from_numpy(U_init.T)
            
        linear_layer.weight = torch.nn.Parameter(U_init)
    
        ortho_layer = torch.nn.utils.parametrizations.orthogonal(linear_layer).to(device)
        assert ortho_layer.weight.shape == (k, d)
        
        optimizer = torch.optim.Adam(ortho_layer.parameters(), lr=lr)
        
        pgb = tqdm(range(epochs), desc=f"{self.__class__.__name__} (k={k})")
        for epoch in pgb:
            optimizer.zero_grad()
            
            # shape: (d, k)
            U = ortho_layer.weight.T

            epsilon = arr_act - ((arr_act @ U)  @ U.T)

            jabobian_epsilon = torch.einsum("ncd,nd->nc", arr_ctx, epsilon)
            
            loss = jabobian_epsilon.pow(2).sum(dim=1)
    
            loss = loss.mean()
            
            loss.backward()
        
            optimizer.step()

            loss = loss.detach().cpu().numpy()
                
            pgb.set_description_str(f"{self.__class__.__name__} (k={k}; lr={lr}) loss={loss:.4e}")
            
        U =  ortho_layer.weight.T.detach().cpu().numpy()
        
        # sanity_check
        np.testing.assert_allclose(U.T @ U, np.eye(k), atol=1e-4)

        return U



class PRCAReconFullJacobian(BasisInterface):
    def __init__(self,  arr_act, arr_ctx, layer=None, task_id=None):
        # todo: extract jacobian
    
        self.arr_act, self.arr_ctx = extract_act_jacobian(model, layer, dl_train)

        assert len(self.arr_ctx.shape) == 3
        
        self.layer = layer
        self.task_id = task_id
        self.is_slow = True
        
    def get_Uk(self, k: int):

        return PRCAReconNonGreedyFullJacobianLearner(
            layer=self.layer,
            task_id=self.task_id
        ).fit(
            self.arr_act, self.arr_ctx, 
            k=k,
            U_init=None,
            device=DEVICE
        )


class PRCAReconFullJacobianPrepro(BasisInterface):
    def __init__(self,  arr_act, arr_ctx, layer=None, task_id=None):
        # todo: extract jacobian
    
        self.arr_act, self.arr_ctx = extract_act_jacobian(model, layer, dl_train)

        assert len(self.arr_ctx.shape) == 3

        _, d = self.arr_act.shape
        self.U_jacobian = JacobianEigenDecom(None, None, layer=layer).get_Uk(d)
        
        self.layer = layer
        self.task_id = task_id
        self.is_slow = True
        
    def get_Uk(self, k: int):

        
        U =  PRCAReconNonGreedyFullJacobianLearner(
            layer=self.layer,
            task_id=self.task_id
        ).fit(
            self.arr_act, self.arr_ctx, 
            k=k,
            U_init=self.U_jacobian[:, :k],
            device=DEVICE
        )
        return U
        

class JacobianEigenDecom(BasisInterface):
    def __init__(self,  arr_act, arr_ctx, layer=None, task_id=None):
        # todo: extract jacobian
    
        self.arr_act, self.arr_ctx = extract_act_jacobian(model, layer, dl_train)

        cov = np.einsum("ncd,ncD->ndD", self.arr_ctx, self.arr_ctx).mean(axis=0)
        np.testing.assert_allclose(
            cov,
            cov.T
        )

        self.U = _solve_eigvecs(cov)
        
        
    def get_Uk(self, k: int):
        return self.U[:, :k]
        
        # return PRCAReconNonGreedyFullJacobianLearner(
        #     layer=self.layer,
        #     task_id=self.task_id
        # ).fit(
        #     self.arr_act, self.arr_ctx, 
        #     k=k,
        #     U_init=None,
        #     device=DEVICE
        # )

# Estimating Accuracy

In [177]:
def construct_fh(Uk):
    def fh(mod, inp, outp):
        return F.conv2d(
            outp,
            (Uk@Uk.T).unsqueeze(2).unsqueeze(3)
        )
    return fh

def compute_task_acc_at_k(
    model, layer,  
    arr_ks,
    output_quantity = LogOddWinningClass(),
    dl_train_from_dataset="cifar100-people",
    arr_basis_classes=[PCA],
    base_output_dir="./artifacts/subclasses",
):

    trng = torch.Generator()
    trng.manual_seed(1)
    
    ds, _ = random_split(
        datasets.construct(dl_train_from_dataset).create_subset(train_split=True),
        [DATASIZE, 1-DATASIZE],
        generator=trng
    )
    dl_train = datasets.build_dataloader(ds, shuffle=False)

    arr_act, arr_ctx = extract_act_grad(
        model, 
        layer, 
        dl=dl_train, 
        output_quantity=output_quantity,
    )

    arr_ks_for_slow_learners = arr_ks
    
    module = getattr(model, layer)

    rel = (arr_act  * arr_ctx).sum(axis=1)

    suffix = "relevance-grad"
    output_path = f"{base_output_dir}/task-{DATASET_NAME}/{layer}/{suffix}"
    os.makedirs(output_path, exist_ok=True)

    arr_dfs = []

    for basis_class in arr_basis_classes:
        basis: BasisInterface = basis_class(
            arr_act=arr_act, 
            arr_ctx=arr_ctx,
            layer=layer,
        )
        basis_name = basis.__class__.__name__
        arr_stat_rows = []

        for k in tqdm(
            arr_ks_for_slow_learners if hasattr(basis, "is_slow") else arr_ks, 
            desc=f"[{basis_name:<20s}] Estimating Performance"
        ):

            Uk = basis.get_Uk(k=k)

            arr_recon_act = (arr_act @ Uk) @ Uk.T
            residue = arr_act - arr_recon_act
            recon_err = np.linalg.norm(residue, axis=1).mean()
            projected_rel = ((arr_act @ Uk) * (arr_ctx @ Uk)).sum(axis=1)
            rel_recon_err = ((rel - projected_rel) **2).mean()
            assert rel.shape == projected_rel.shape == (rel.shape[0], )
            perc_sign_align = (np.sign(rel) * np.sign(projected_rel)).mean()
            cosine = ( 
                (arr_ctx / (np.linalg.norm(arr_ctx, axis=1, keepdims=True)) + 1e-6) \
                * (residue / (np.linalg.norm(residue, axis=1, keepdims=True)  + 1e-6))
            ).sum(axis=1)
            cosine = np.abs(cosine).mean()

            recon_loss = ((arr_ctx * residue).sum(axis=1) ** 2).mean()
            Uk = torch.from_numpy(Uk).to(DEVICE)
            row = dict(
                layer=layer,
                dl_train_from_dataset=dl_train_from_dataset,
                output_quantity=output_quantity.__class__.__name__,
                k=k, 
                basis_name=basis_name,
                recon_err=recon_err,
                cosine=cosine,
                rel_recon_err=rel_recon_err,
                perc_sign_align=perc_sign_align,
            )
            try:
                hook = module.register_forward_hook(construct_fh(Uk))
                
                for label, dl in [
                    ("val", dl_val)
                ]:
                    row[f"acc_{label}"], row[f"xent_{label}"] = metrics.accuracy(
                        model=model,
                        dataloader=dl,
                        num_classes=len(dataset.selected_classes),
                        device=DEVICE,
                    )
                
            finally:
                hook.remove()
            arr_stat_rows.append(row)
            
        df = pd.DataFrame(arr_stat_rows)
        df.to_csv(
            f"{output_path}/{basis_name}.csv",
            index=False
        )

        arr_dfs.append(df)

    df = pd.concat(arr_dfs).sort_values(by=["k", "acc_val"], ascending=[True, False])
    print(f"Checking results at {output_path}")
    return df


# logit_mod_list = winning-class, pos-winner-neg-others, pos-winner-neg-others-onehot, diff-top2winner, all-classes
    

compute_task_acc_at_k(
    model, layer="avgpool", 
    arr_ks=[1, 2, 5],
    base_output_dir="./tmp",
    dl_train_from_dataset="cifar100-people",
    arr_basis_classes=[
        PRCAReconFullJacobianPrepro,
        
        JacobianEigenDecom,
        PRCAReconFullJacobian,
        # # PRCARecon,
        # # PRCAReconGreedy,
        # # PRCAReconNormGrad,
        # # PCA,
        # GradPCA,
        # PRCA,
        # PRCASortAbs,
        # PRCARecon
     ]
)

extract act ctx output_quantity=LogOddWinningClass) at layer=avgpool:   0%|          | 0/4 [00:00<?, ?it/s]

avgpool: output-dims=torch.Size([512, 1, 1])


extract act ctx output_quantity=LogitOfOutput) at layer=avgpool:   0%|          | 0/4 [00:00<?, ?it/s]

avgpool: output-dims=torch.Size([512, 1, 1])


extract act ctx output_quantity=LogitOfOutput) at layer=avgpool:   0%|          | 0/4 [00:00<?, ?it/s]

...

PRCAReconNonGreedyFullJacobianLearner (k=1):   0%|          | 0/5000 [00:00<?, ?it/s]

PRCAReconNonGreedyFullJacobianLearner (k=2):   0%|          | 0/5000 [00:00<?, ?it/s]

PRCAReconNonGreedyFullJacobianLearner (k=5):   0%|          | 0/5000 [00:00<?, ?it/s]

Checking results at ./tmp/task-cifar100-people/avgpool/relevance-grad


,layer,dl_train_from_dataset,output_quantity,k,basis_name,recon_err,cosine,rel_recon_err,perc_sign_align,acc_val,xent_val
0,avgpool,cifar100-people,LogOddWinningClass,1,JacobianEigenDecom,11.061205,0.038868,1.325157e+01,1.0,0.202,1.666957
0,avgpool,cifar100-people,LogOddWinningClass,1,PRCAReconFullJacobianPrepro,10.908578,0.038009,1.327037e+01,1.0,0.200,1.636544
0,avgpool,cifar100-people,LogOddWinningClass,1,PRCAReconFullJacobian,10.917809,0.037982,1.337248e+01,1.0,0.200,1.636501
1,avgpool,cifar100-people,LogOddWinningClass,2,JacobianEigenDecom,10.319324,0.021611,2.747885e+00,1.0,0.400,1.315305
1,avgpool,cifar100-people,LogOddWinningClass,2,PRCAReconFullJacobian,10.144357,0.018280,2.826585e+00,1.0,0.370,1.456833
1,avgpool,cifar100-people,LogOddWinningClass,2,PRCAReconFullJacobianPrepro,10.146613,0.018263,2.829425e+00,1.0,0.368,1.457038
2,avgpool,cifar100-people,LogOddWinningClass,5,PRCAReconFullJacobianPrepro,8.915315,0.000859,1.210676e-03,1.0,0.620,1.295083
2,avgpool,cifar100-people,LogOddWinningClass,5,JacobianEigenDecom,8.935152,0.000018,1.726585e-11,1.0,0.620,1.294872
2,avgpool,cifar100-people,LogOddWinningClass,5,PRCAReconFullJacobian,8.873291,0.000170,6.568805e-05,1.0,0.614,1.292535


In [170]:
# compute_task_acc_at_k(
#     model, layer="layer3", 
#     arr_ks=[2, 5, 10],
#     base_output_dir="./tmp",
#     dl_train_from_dataset="cifar100-people",
#     arr_basis_classes=[
#         JacobianEigenDecom,
#         PRCAReconFullJacobian,
#         PRCARecon,
#         # PRCAReconGreedy,
#         # PRCAReconNormGrad,
#         PCA,
#         GradPCA,
#         # PRCA,
#         # PRCASortAbs,
#         # PRCARecon
#      ]
# )

In [178]:

compute_task_acc_at_k(
    model, layer="layer3", 
    arr_ks=[10, 20, 30],
    base_output_dir="./tmp",
    dl_train_from_dataset="cifar100",
    arr_basis_classes=[
        PRCAReconFullJacobianPrepro,
        JacobianEigenDecom,
        PRCAReconFullJacobian,
        
        # PRCARecon,
        # # PRCAReconGreedy,
        # # PRCAReconNormGrad,
        # PCA,
        # GradPCA,
        # PRCA,
        # PRCASortAbs,
        # PRCARecon
     ]
)

extract act ctx output_quantity=LogOddWinningClass) at layer=layer3:   0%|          | 0/79 [00:00<?, ?it/s]

layer3: output-dims=torch.Size([256, 8, 8])


extract act ctx output_quantity=LogitOfOutput) at layer=layer3:   0%|          | 0/4 [00:00<?, ?it/s]

layer3: output-dims=torch.Size([256, 8, 8])


extract act ctx output_quantity=LogitOfOutput) at layer=layer3:   0%|          | 0/4 [00:00<?, ?it/s]

...

PRCAReconNonGreedyFullJacobianLearner (k=10):   0%|          | 0/5000 [00:00<?, ?it/s]

PRCAReconNonGreedyFullJacobianLearner (k=20):   0%|          | 0/5000 [00:00<?, ?it/s]

PRCAReconNonGreedyFullJacobianLearner (k=30):   0%|          | 0/5000 [00:00<?, ?it/s]

Checking results at ./tmp/task-cifar100-people/layer3/relevance-grad


,layer,dl_train_from_dataset,output_quantity,k,basis_name,recon_err,cosine,rel_recon_err,perc_sign_align,acc_val,xent_val
0,layer3,cifar100,LogOddWinningClass,10,JacobianEigenDecom,1.538968,0.079849,0.072364,0.17374,0.486,2.631901
0,layer3,cifar100,LogOddWinningClass,10,PRCAReconFullJacobianPrepro,1.544260,0.062443,0.033302,0.45552,0.422,1.458317
0,layer3,cifar100,LogOddWinningClass,10,PRCAReconFullJacobian,1.544260,0.062443,0.033302,0.45554,0.422,1.458136
1,layer3,cifar100,LogOddWinningClass,20,JacobianEigenDecom,1.427114,0.072366,0.045567,0.37988,0.574,1.859043
1,layer3,cifar100,LogOddWinningClass,20,PRCAReconFullJacobianPrepro,1.450890,0.055547,0.022359,0.56434,0.562,1.250073
1,layer3,cifar100,LogOddWinningClass,20,PRCAReconFullJacobian,1.450897,0.055545,0.022358,0.56438,0.562,1.250157
2,layer3,cifar100,LogOddWinningClass,30,PRCAReconFullJacobianPrepro,1.361972,0.050955,0.016843,0.63612,0.590,1.216458
2,layer3,cifar100,LogOddWinningClass,30,JacobianEigenDecom,1.354230,0.061903,0.028773,0.53750,0.588,1.643654
2,layer3,cifar100,LogOddWinningClass,30,PRCAReconFullJacobian,1.361988,0.050954,0.016843,0.63598,0.588,1.216720


In [179]:
compute_task_acc_at_k(
    model, layer="layer2", 
    arr_ks=[10, 20, 30],
    base_output_dir="./tmp",
    dl_train_from_dataset="cifar100",
    arr_basis_classes=[
        JacobianEigenDecom,
        PRCAReconFullJacobian,
        PRCAReconFullJacobianPrepro,
        # PCA,
        # GradPCA,
     ]
)

extract act ctx output_quantity=LogOddWinningClass) at layer=layer2:   0%|          | 0/79 [00:00<?, ?it/s]

layer2: output-dims=torch.Size([128, 16, 16])


extract act ctx output_quantity=LogitOfOutput) at layer=layer2:   0%|          | 0/4 [00:00<?, ?it/s]

layer2: output-dims=torch.Size([128, 16, 16])


extract act ctx output_quantity=LogitOfOutput) at layer=layer2:   0%|          | 0/4 [00:00<?, ?it/s]

...

PRCAReconNonGreedyFullJacobianLearner (k=10):   0%|          | 0/5000 [00:00<?, ?it/s]

PRCAReconNonGreedyFullJacobianLearner (k=20):   0%|          | 0/5000 [00:00<?, ?it/s]

PRCAReconNonGreedyFullJacobianLearner (k=30):   0%|          | 0/5000 [00:00<?, ?it/s]

Checking results at ./tmp/task-cifar100-people/layer2/relevance-grad


,layer,dl_train_from_dataset,output_quantity,k,basis_name,recon_err,cosine,rel_recon_err,perc_sign_align,acc_val,xent_val
0,layer2,cifar100,LogOddWinningClass,10,PRCAReconFullJacobianPrepro,1.450125,0.084099,0.004319,0.65402,0.476,1.549136
0,layer2,cifar100,LogOddWinningClass,10,PRCAReconFullJacobian,1.450122,0.084094,0.004319,0.65390,0.472,1.548962
0,layer2,cifar100,LogOddWinningClass,10,JacobianEigenDecom,1.645548,0.091052,0.006320,0.56308,0.374,1.582412
1,layer2,cifar100,LogOddWinningClass,20,PRCAReconFullJacobianPrepro,1.205173,0.064262,0.001718,0.78816,0.606,1.455365
1,layer2,cifar100,LogOddWinningClass,20,PRCAReconFullJacobian,1.196742,0.064173,0.001684,0.78880,0.590,1.465087
1,layer2,cifar100,LogOddWinningClass,20,JacobianEigenDecom,1.513976,0.076211,0.003602,0.68436,0.464,1.529915
2,layer2,cifar100,LogOddWinningClass,30,JacobianEigenDecom,1.297780,0.054715,0.001324,0.81236,0.630,1.376627
2,layer2,cifar100,LogOddWinningClass,30,PRCAReconFullJacobian,1.034666,0.049385,0.000728,0.86330,0.626,1.373164
2,layer2,cifar100,LogOddWinningClass,30,PRCAReconFullJacobianPrepro,1.035027,0.049372,0.000728,0.86324,0.626,1.373403


In [180]:
compute_task_acc_at_k(
    model, layer="layer1", 
    arr_ks=[10, 20, 30],
    base_output_dir="./tmp",
    dl_train_from_dataset="cifar100",
    arr_basis_classes=[
        JacobianEigenDecom,
        PRCAReconFullJacobian,
        PRCAReconFullJacobianPrepro,
        # PCA,
        # GradPCA,
     ]
)

extract act ctx output_quantity=LogOddWinningClass) at layer=layer1:   0%|          | 0/79 [00:00<?, ?it/s]

layer1: output-dims=torch.Size([64, 32, 32])


extract act ctx output_quantity=LogitOfOutput) at layer=layer1:   0%|          | 0/4 [00:00<?, ?it/s]

layer1: output-dims=torch.Size([64, 32, 32])


extract act ctx output_quantity=LogitOfOutput) at layer=layer1:   0%|          | 0/4 [00:00<?, ?it/s]

...

PRCAReconNonGreedyFullJacobianLearner (k=10):   0%|          | 0/5000 [00:00<?, ?it/s]

PRCAReconNonGreedyFullJacobianLearner (k=20):   0%|          | 0/5000 [00:00<?, ?it/s]

PRCAReconNonGreedyFullJacobianLearner (k=30):   0%|          | 0/5000 [00:00<?, ?it/s]

Checking results at ./tmp/task-cifar100-people/layer1/relevance-grad


,layer,dl_train_from_dataset,output_quantity,k,basis_name,recon_err,cosine,rel_recon_err,perc_sign_align,acc_val,xent_val
0,layer1,cifar100,LogOddWinningClass,10,PRCAReconFullJacobian,0.887201,0.081238,0.000289,0.84248,0.580,1.494856
0,layer1,cifar100,LogOddWinningClass,10,PRCAReconFullJacobianPrepro,0.887210,0.081239,0.000289,0.84254,0.580,1.494719
0,layer1,cifar100,LogOddWinningClass,10,JacobianEigenDecom,1.324280,0.089655,0.000659,0.73814,0.384,1.606632
1,layer1,cifar100,LogOddWinningClass,20,JacobianEigenDecom,0.903104,0.056155,0.000088,0.88732,0.626,1.314894
1,layer1,cifar100,LogOddWinningClass,20,PRCAReconFullJacobian,0.509597,0.045910,0.000027,0.94990,0.616,1.307139
1,layer1,cifar100,LogOddWinningClass,20,PRCAReconFullJacobianPrepro,0.510884,0.045600,0.000027,0.95056,0.614,1.307670
2,layer1,cifar100,LogOddWinningClass,30,PRCAReconFullJacobian,0.351452,0.034367,0.000007,0.97472,0.614,1.297000
2,layer1,cifar100,LogOddWinningClass,30,PRCAReconFullJacobianPrepro,0.348484,0.033671,0.000007,0.97506,0.614,1.291782
2,layer1,cifar100,LogOddWinningClass,30,JacobianEigenDecom,0.671779,0.037737,0.000019,0.94384,0.612,1.292812


In [21]:
print(f"Finished at {datetime.now()}")

Finished at 2025-01-17 09:08:45.254382
